# Bronze Layer

Download energy batch data from AEMO files.

In [0]:
import requests
import os
import time
from datetime import datetime

SUCCESS = 200
MONTHS = range(1, 13)
CURRENT_YEAR = datetime.now().year
YEARS = range(2020, CURRENT_YEAR)  # exclude incomplete current year
REGION = "VIC1"
BRONZE_PATH = "/Volumes/workspace/default/bronze/aemo_raw"

def download_aemo_month(year, month, region):
    url = f"https://www.aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_{year}{month:02d}_{region}.csv"
    response = requests.get(url)

    if response.status_code == SUCCESS:
        return response.text
    else:
        print(f"failed: {url} {response.status_code}")
        return None

def main():
    os.makedirs(BRONZE_PATH, exist_ok=True)

    for year in YEARS:
        for month in MONTHS:
            file_path = f"{BRONZE_PATH}/{year}_{month:02d}_{REGION}.csv"

            if os.path.exists(file_path):
                print(f"{year}-{month:02d} already exists")
                continue

            print(f"fetching {year}-{month:02d}...")
            csv_text = download_aemo_month(year, month, REGION)

            if csv_text:
                with open(file_path, "w") as f:
                    f.write(csv_text)

            time.sleep(0.5)

    print("[BRONZE] batch data downloaded")

## Ingestion

Run the `main` entrypoint to ingest data or run the testing cell to see if the functions work on a small subset before processing the full dataset.

### Testing

In [0]:
test = download_aemo_month(2024, 1, REGION)
print(test[:500])

### Running

Running `main` will take a few minutes to complete. It will save the data instead of printing it.

In [0]:
main()